## Fetch the Genres from OpenLibrary

In [ ]:
import requests
import pandas as pd
from concurrent.futures import ThreadPoolExecutor, as_completed
from tqdm import tqdm

In [ ]:
df = pd.read_csv("/content/books_isbn_title.csv")

In [ ]:
# List or DataFrame of ISBNs
df.head()

,ISBN,title
0,0195153448,Classical Mythology
1,0002005018,Clara Callan
2,0060973129,Decision in Normandy
3,0374157065,Flu: The Story of the Great Influenza Pandemic...
4,0393045218,The Mummies of Urumchi


In [ ]:
isbns = df.copy()
isbns.head()

,ISBN,title
0,0195153448,Classical Mythology
1,0002005018,Clara Callan
2,0060973129,Decision in Normandy
3,0374157065,Flu: The Story of the Great Influenza Pandemic...
4,0393045218,The Mummies of Urumchi


In [ ]:
isbns.isna().sum()

,0
ISBN,0
title,0


In [ ]:
isbns.duplicated().sum()

np.int64(0)

In [ ]:
isbns.shape

(271360, 2)

In [ ]:
# Get the metadata in bulk (json)
!wget https://openlibrary.org/data/ol_dump_editions_latest.txt.gz

--2025-06-15 11:15:44--  https://openlibrary.org/data/ol_dump_editions_latest.txt.gz
Resolving openlibrary.org (openlibrary.org)... 207.241.234.205
Connecting to openlibrary.org (openlibrary.org)|207.241.234.205|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://archive.org/download/ol_dump_2025-05-31/ol_dump_editions_2025-05-31.txt.gz [following]
--2025-06-15 11:15:45--  https://archive.org/download/ol_dump_2025-05-31/ol_dump_editions_2025-05-31.txt.gz
Resolving archive.org (archive.org)... 207.241.224.2
Connecting to archive.org (archive.org)|207.241.224.2|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://ia800300.us.archive.org/22/items/ol_dump_2025-05-31/ol_dump_editions_2025-05-31.txt.gz [following]
--2025-06-15 11:15:46--  https://ia800300.us.archive.org/22/items/ol_dump_2025-05-31/ol_dump_editions_2025-05-31.txt.gz
Resolving ia800300.us.archive.org (ia800300.us.archive.org)... 207.241.228.10
Connecting to i

In [ ]:
import gzip

file_path = '/content/ol_dump_editions_latest.txt.gz'

with gzip.open(file_path, 'rt', encoding='utf-8') as f:
    for _ in range(5):  # read first 10 lines
        print(f.readline())


/type/edition	/books/OL10000079M	2	2010-03-12T00:00:48.298004	{"publishers": ["Stationery Office Books"], "physical_format": "Paperback", "subjects": ["Central government", "United Kingdom, Great Britain"], "created": {"type": "/type/datetime", "value": "2008-04-30T09:38:13.731961"}, "isbn_10": ["010771762X"], "number_of_pages": 10, "isbn_13": ["9780107717629"], "last_modified": {"type": "/type/datetime", "value": "2010-03-12T00:00:48.298004"}, "publish_date": "March 31, 1999", "key": "/books/OL10000079M", "authors": [{"key": "/authors/OL46053A"}], "title": "Index to the House of Lords Parliamentary Debates", "latest_revision": 2, "works": [{"key": "/works/OL14903346W"}], "type": {"key": "/type/edition"}, "revision": 2}

/type/edition	/books/OL10000715M	3	2011-04-26T05:54:13.372021	{"publishers": ["Stationery Office Books"], "physical_format": "Paperback", "last_modified": {"type": "/type/datetime", "value": "2011-04-26T05:54:13.372021"}, "title": "Northern Ireland (Emergency Provision

In [ ]:
import gzip
import json
import csv
from multiprocessing import Pool, cpu_count

file_path = '/content/ol_dump_editions_latest.txt.gz'
output_file = '/content/fast_filtered_subjects.csv'

def process_line(line):
    try:
        json_blob = line.strip().split('\t')[-1]
        record = json.loads(json_blob)
        isbn_10 = record.get('isbn_10', [])
        subjects = record.get('subjects', [])
        if not isbn_10:
            return []
        return [{'isbn_10': isbn, 'subjects': '|'.join(subjects)} for isbn in isbn_10]
    except:
        return []

def process_in_chunks():
    with gzip.open(file_path, 'rt', encoding='utf-8') as f, \
         open(output_file, 'w', newline='', encoding='utf-8') as csvfile:

        writer = csv.DictWriter(csvfile, fieldnames=['isbn_10', 'subjects'])
        writer.writeheader()

        pool = Pool(cpu_count())

        batch = []
        for i, line in enumerate(f):
            batch.append(line)
            if len(batch) >= 10000:
                results = pool.map(process_line, batch)
                for row_list in results:
                    for row in row_list:
                        writer.writerow(row)
                batch = []
                print(f"Processed {i} lines...")

        # Final batch
        if batch:
            results = pool.map(process_line, batch)
            for row_list in results:
                for row in row_list:
                    writer.writerow(row)

        pool.close()
        pool.join()

process_in_chunks()


Streaming output truncated to the last 5000 lines.
Processed 4309999 lines...
Processed 4319999 lines...
Processed 4329999 lines...
Processed 4339999 lines...
Processed 4349999 lines...
Processed 4359999 lines...
Processed 4369999 lines...
Processed 4379999 lines...
Processed 4389999 lines...
Processed 4399999 lines...
Processed 4409999 lines...
Processed 4419999 lines...
Processed 4429999 lines...
Processed 4439999 lines...
Processed 4449999 lines...
Processed 4459999 lines...
Processed 4469999 lines...
Processed 4479999 lines...
Processed 4489999 lines...
Processed 4499999 lines...
Processed 4509999 lines...
Processed 4519999 lines...
Processed 4529999 lines...
Processed 4539999 lines...
Processed 4549999 lines...
Processed 4559999 lines...
Processed 4569999 lines...
Processed 4579999 lines...
Processed 4589999 lines...
Processed 4599999 lines...
Processed 4609999 lines...
Processed 4619999 lines...
Processed 4629999 lines...
Processed 4639999 lines...
Processed 4649999 lines...
Proc

In [ ]:
subjects_df = pd.read_csv("/content/filtered_isbn_subjects.csv")

In [ ]:
subjects_df.sample(10)

,isbn_10,subjects
17072175,5941280171,NaN
6298991,0101845103,Great Britain. -- Parliament. -- House of Comm...
12865177,0738854611,Biography: general|General|Biography & Autobio...
11404292,0944622062,Latvians -- Australia.
10744007,0921100884,Protestantism -- History.
10948096,0873512162,"Dakota Indians -- Wars, 1862-1865 -- Personal ..."
13213176,089742039X,"Science fiction, American.|Children's stories,..."
2594673,1315648830,Medical informatics|Telecommunication in medic...
469039,5937860500,"Quotations, Russian -- Dictionaries.|Arts -- Q..."
12138253,0662992601,Entreprises étrangères -- Canada.


In [ ]:
subjects_df.rename(columns={'isbn_10': 'ISBN'}, inplace=True)

In [ ]:
subjects_df.head()

,ISBN,subjects
0,010771762X,"Central government|United Kingdom, Great Britain"
1,0108366251,NaN
2,0108380270,English law: criminal law
3,0108383784,NaN
4,0891458875,Quilting -- Patterns.


In [ ]:
isbns.head()

,ISBN,title
0,0195153448,Classical Mythology
1,0002005018,Clara Callan
2,0060973129,Decision in Normandy
3,0374157065,Flu: The Story of the Great Influenza Pandemic...
4,0393045218,The Mummies of Urumchi


In [ ]:
genres = isbns.merge(subjects_df, how='left', on='ISBN')

In [ ]:
genres.head()

,ISBN,title,subjects
0,0195153448,Classical Mythology,"Mythology, Classical."
1,0195153448,Classical Mythology,"Mythology, Classical"
2,0195153448,Classical Mythology,NaN
3,0002005018,Clara Callan,NaN
4,0002005018,Clara Callan,Women teachers -- Fiction.|Young women -- Fict...


In [ ]:
genres.isna().sum()


,0
ISBN,0
title,0
subjects,117738


In [ ]:

genres.duplicated().sum()

np.int64(19967)

In [ ]:
genres.shape

(371685, 3)

In [ ]:
# Define a helper function to detect empty or "useless" subjects
def is_invalid_subject(x):
    if pd.isna(x):
        return True
    if isinstance(x, str) and x.strip() == "":
        return True
    return False

# Step 1: Mark invalid rows
genres['is_invalid'] = genres['subjects'].apply(is_invalid_subject)

# Step 2: Sort so valid rows come first
genres_sorted = genres.sort_values(by='is_invalid')

# Step 3: Drop duplicates, keeping the first (valid if exists)
genres_dedup = genres_sorted.drop_duplicates(subset=['ISBN', 'title'], keep='first')

# Step 4: Drop the helper column
genres_dedup = genres_dedup.drop(columns=['is_invalid'])


In [ ]:
genres_dedup.sample(20)

,ISBN,title,subjects
119213,3442552141,"Die Katze, die nach Paris reiste.",NaN
370510,0701137819,Passions of the Mind,Literary studies: general
351889,0590738704,Stephen Biesty's Incredible Cross-Sections Book,Rocks -- Collection and preservation.|Minerals...
275982,0140064176,Further Down on Maggie's Farm,NaN
42855,0684189259,HE HUFFED AND HE PUFFED,NaN
230474,044991173X,"The Hollow Hills (Stewart, Mary, Arthurian Sag...","Arthur, King -- Fiction.|Merlin (Legendary cha..."
143358,0696206137,Make-It-Simple Entertaining: Fabulous Menus fo...,Menus|Cooking / Wine|Cooking|Entertaining - Ge...
290521,1892714000,The Ripple Effect: Our Harvest,"Spirituality|New Age|Body, Mind & Spirit|Alter..."
15447,0156106809,The Baron in the Trees,NaN
136399,0671874985,Where Love Has Gone,Fiction - General|Fiction / General|Fiction


In [ ]:
genres_clean = genres_dedup.copy()

In [ ]:
import re

def clean_subjects(text):
    if pd.isna(text):
        return None

    # Remove parentheses and their content
    text = re.sub(r'\(.*?\)', '', text)

    # Replace common delimiters with '|'
    text = re.sub(r'[|,]', ' | ', text)

    # Remove periods and ellipses
    text = re.sub(r'\.+', '', text)

    # Split, clean, and filter
    parts = [p.strip().lower() for p in text.split('|') if p.strip()]

    simplified = []
    seen = set()
    for part in parts:
        # Take only the first token before dash/colon
        keyword = re.split(r'\s*[-:]\s*', part)[0].title()

        # Filter out numeric-only tokens
        if keyword and not keyword.isdigit() and keyword not in seen:
            seen.add(keyword)
            simplified.append(keyword)

    # Limit to 3 keywords
    return ' | '.join(simplified[:3]) if simplified else None

In [ ]:
genres_clean['subjects'] = genres_clean['subjects'].apply(clean_subjects)

In [ ]:
genres_clean.sample(20)

,ISBN,title,subjects
338155,0670490032,Mother Goose Nursery Rhymes (Studio Book),Nursery Rhymes
106741,0679601759,Thus Spoke Zarathustra : A Book for All and No...,Superman | Philosophy
191818,0880386495,Greyhawk Adventures (Advanced Dungeons &amp; D...,Dungeons And Dragons
44636,0671004271,The Evening Star,Movie/Tv Tie | Fiction | Sagas
27265,0679886370,Stargirl,Individuality | Popularity | Eccentrics And Ec...
135572,0553157388,Encyclopedia Brown and the Case of the Midnigh...,Detective And Mystery Stories
125762,0440137632,Hotel,None
22194,8401490693,Arena y viento (Los Jet de Plaza &amp; JanÃ©s),Modern Fiction
189454,0020420307,Abraham Lincoln : The Great Emancipator (Child...,Lincoln | Abraham | Presidents
285383,0520209087,Backstory 2: Interviews With Screenwriters of ...,Film Theory & Criticism | Other Prose | Plays ...


In [ ]:
result = genres_clean.loc[genres_clean['ISBN'] == '0736410457', 'cleaned_subjects']
print(result.values)

['Social Issues | Family | Juvenile Fiction']


In [ ]:
genres_clean.to_csv("genres_clean.csv", index=False)
from google.colab import files
files.download("genres_clean.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## Apply the custom genres to the Null fields (using PyTorch and pre-trained NLI)



In [3]:
!pip install torch torchvision torchaudio --quiet
!pip install transformers --quiet

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 126.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 93.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 55.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 14.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 7.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 6.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 45.8 MB/s eta 0:00:00


In [4]:
import torch
print("CUDA available:", torch.cuda.is_available())
print("Device name:", torch.cuda.get_device_name(0))


CUDA available: True
Device name: Tesla T4


In [2]:
# 1) Libraris
from transformers import pipeline
from tqdm import tqdm
import pandas as pd


In [5]:
clean_df = pd.read_csv("/content/genres_clean.csv")

/tmp/ipython-input-5-3409312415.py:1: DtypeWarning: Columns (2) have mixed types. Specify dtype option on import or set low_memory=False.
  clean_df = pd.read_csv("/content/genres_clean.csv")


In [ ]:
# Convert None to NaN (for pandas compatibility)
#clean_df['subjects'] = clean_df['subjects'].replace({None: pd.NA})

In [ ]:
clean_df.sample(10)

,ISBN,title,subjects
1247,0898041414,Far Memory (Joan Grant Autobiography),Grant | Joan Marshall | Authors
189730,0451143353,Harem,Fiction / General | Movie/Tv Tie
219674,0394507320,"Shikasta: Re, Colonized Planet 5 : Personal, P...",NaN
248498,8877671343,La citta' sfiorita: Romanzo (Fantasia e memoria),NaN
152169,1887166440,"The Inner Bitch Guide to Men, Relationships, D...",Women | Man | Dating
269641,0451175859,Critical Condition,NaN
244489,0505524856,The Shadow Prince (Heartspell),NaN
225650,006080646X,"Protocol for a Kidnapping (Perennial Library,)",NaN
153274,0849914663,Time To Pray Series: Prayers For Bedtime,Bedtime Prayers | Children | Toy And Movable B...
75614,095342054X,Yan and the Pike: A Very Strange Tale About A ...,Modern Fiction | Fiction


In [6]:
# Filter subjects with exactly 2 characters
short_subjects = clean_df["subjects"].dropna().apply(lambda x: x.strip()).loc[lambda x: x.str.len() <= 3]

# Get the count
count = short_subjects.shape[0]
print(f"Number of subject values with exactly 2 characters: {count}")

# Show unique values
unique_vals = short_subjects.unique()
print("Unique 2-character subject values:")
print(unique_vals)


Number of subject values with exactly 2 characters: 684
Unique 2-character subject values:
['C' 'God' 'Sex' 'Man' 'C++' 'Law' 'Non' 'Mad' 'Men' 'Co' 'Art' 'Low'
 'Sat' 'Xml' 'Yo' 'Elt' 'Go' 'Sql' 'Ms' 'Air' 'Er' 'X' 'C&&' 'Tao' 'Joy'
 'In' 'One' 'Cd' 'Ik' 'Sun' 'Red' 'Php' 'Gw' 'Cid' 'Ml' 'Qi' 'Cgi' 'Ya'
 'Lao' 'Tea' 'Neo' 'C₊₊' 'Rpg' 'Aix' 'Yak' 'Xsl' 'Rap' 'Ex' 'Pi' 'San'
 'Oak' 'Xi' 'U2' 'E' 'Mig' 'Pre' 'V 1']


In [7]:
# Filter subjects with exactly 2 characters
short_subjects = clean_df["subjects"].dropna().apply(lambda x: x.strip()).loc[lambda x: x.str.len() <= 2]

# Get the count
count = short_subjects.shape[0]
print(f"Number of subject values with exactly 2 characters: {count}")

# Show unique values
unique_vals = short_subjects.unique()
print("Unique 2-character subject values:")
print(unique_vals)


Number of subject values with exactly 2 characters: 62
Unique 2-character subject values:
['C' 'Co' 'Yo' 'Go' 'Ms' 'Er' 'X' 'In' 'Cd' 'Ik' 'Gw' 'Ml' 'Qi' 'Ya' 'Ex'
 'Pi' 'Xi' 'U2' 'E']


In [8]:
# Define unwanted short subject values

# List of bad labels to remove
bad_labels = {
    "C", "Non", "Co", "Low", "Ms", "Er", "X", "C&&", "In", "One", "Cd", "Ik",
    "Red", "Gw", "Cid", "Cgi", "Ya", "Yak", "Xsl", "Rap", "Ex", "Pi", "San",
    "Xi", "U2", "E", "Mig", "Pre", "V 1", 'V V', 'Etc', 'Self'
}

# Function to clean multi-label subject strings
def clean_subjects(subject_string):
    if pd.isna(subject_string):
        return pd.NA
    # Split, strip, filter bad labels
    labels = [label.strip() for label in str(subject_string).split("|")]
    cleaned = [label for label in labels if label not in bad_labels]
    return " | ".join(cleaned) if cleaned else pd.NA

# Apply to column
clean_df["subjects"] = clean_df["subjects"].apply(clean_subjects)


In [9]:
# Filter subjects with exactly 2 characters
short_subjects = clean_df["subjects"].dropna().apply(lambda x: x.strip()).loc[lambda x: x.str.len() <= 3]

# Get the count
count = short_subjects.shape[0]
print(f"Number of subject values with exactly 2 characters: {count}")

# Show unique values
unique_vals = short_subjects.unique()
print("Unique 2-character subject values:")
print(unique_vals)


Number of subject values with exactly 2 characters: 330
Unique 2-character subject values:
['God' 'Sex' 'Man' 'C++' 'Law' 'Mad' 'Men' 'Art' 'Sat' 'Xml' 'Yo' 'Elt'
 'Go' 'Sql' 'Air' 'Tao' 'Joy' 'Sun' 'Php' 'Ml' 'Qi' 'Lao' 'Tea' 'Neo'
 'C₊₊' 'Rpg' 'Aix' 'Hip' 'Pc' 'Dos' 'Oak']


In [10]:
# Filter rows where 'subjects' contains any digit
subjects_with_digits = clean_df[clean_df["subjects"].str.contains(r"\d", na=False)]

# Count the number of such rows
print(f"Number of subject entries containing digits: {len(subjects_with_digits)}")

# Get unique values and their frequencies
digit_subject_counts = subjects_with_digits["subjects"].value_counts()

# Display the result
print("Unique subject values containing digits and their counts:")
print(digit_subject_counts)


Number of subject entries containing digits: 6477
Unique subject values containing digits and their counts:
subjects
General | Children'S 9                                             84
General | Children'S 9 | Juvenile Fiction                          66
General | Juvenile Fiction / General | Children'S 9                63
19Th Century Fiction | Literature | Fiction                        63
19Th Century Fiction                                               55
                                                                   ..
19Th Century Fiction | Horror & Ghost Stories | Novels              1
19Th Century Fiction | Short Stories | Austen                       1
Juvenile Mysteries | Juvenile Fiction | Children'S Books/Ages 4     1
Biography | Postwar Period | 1945 To C 2000                         1
19Th Century Fiction | Literary Studies | Sale Adult                1
Name: count, Length: 3857, dtype: int64


In [11]:
import re

# Compile the patterns you want to keep
allowed_digit_patterns = [
    re.compile(r"children'?s\s*\d{1,2}", re.IGNORECASE),            # Children's 9, Children's 12, etc.
    re.compile(r"\d{1,2}th\s+century", re.IGNORECASE)
]

def is_allowed_digit_phrase(part):
    """Check if part matches any allowed pattern."""
    return any(pattern.fullmatch(part.strip()) for pattern in allowed_digit_patterns)

def clean_subjects_digits(value):
    """Remove digit-containing segments unless they match allowed patterns."""
    if pd.isna(value):
        return value
    parts = [p.strip() for p in value.split("|")]
    cleaned = []
    for part in parts:
        if re.search(r"\d", part):  # contains digits
            if is_allowed_digit_phrase(part):
                cleaned.append(part.strip())
            # else → skip
        else:
            cleaned.append(part.strip())
    return " | ".join(cleaned) if cleaned else pd.NA

# Apply to DataFrame
clean_df["subjects"] = clean_df["subjects"].apply(clean_subjects_digits)


In [ ]:
clean_df.sample(15)

,ISBN,title,subjects
84533,0966351908,Sugar Bust for Life!... With the Brennans: Coo...,Recipes | Health & Healing | Courses & Dishes
140974,3423125004,Das Kartengesheimnis,Language Readers | Literature | Untranslated F...
51816,0811829510,The Red Thread: A Love Story,Shanghai
125929,1575661195,The Yellow Room,Modern Fiction | Mystery & Detective | Fiction
156348,0395198437,"Money, Whence It Came, Where It Went",Money | Economic History
194536,0373196644,Captivating A Cowboy,Romance | Fiction / Romance / Adult | Fiction
248809,0374177694,After Jihad: America and the Struggle for Isla...,<NA>
184476,0373169639,Sweeping the Bride Away (Harlequin American Ro...,Fiction | Romance | Fiction / Romance / Adult
123960,1557487952,Inspirational Romance Reader: Contemporary Col...,Religion | Fiction | Religious
47172,1400032520,The Blood Doctor : A Novel (Vintage Crime/Blac...,Physicians | Great Britain


In [12]:
from collections import Counter
# Show all rows in the DataFrame output
pd.set_option('display.max_rows', None)

# Assuming clean_df is your DataFrame

# 1. Drop NA rows in subjects (optional, or you can fillna with empty string)
subjects_series = clean_df["subjects"].dropna()

# 2. Split each subject string by "|" and flatten into a list of all phrases
all_phrases = subjects_series.str.split(r"\s*\|\s*").explode().str.strip()

# 3. Count occurrences of each phrase
phrase_counts = Counter(all_phrases)

# 4. Get the 100 most common phrases as a list of (phrase, count) tuples
top_phrases = phrase_counts.most_common(200)

# Optional: convert to DataFrame for nicer display or further use
subject_phrases = pd.DataFrame(top_phrases, columns=["phrase", "count"])

print(subject_phrases)


                                      phrase  count
0                                    Fiction  37655
1                                    General  13854
2                                    Romance  12069
3                             Modern Fiction   6138
4                            Science Fiction   5923
5                                    Fantasy   4004
6                        Mystery & Detective   3200
7                              United States   3187
8                                   Children   2904
9                                   American   2882
10                          Juvenile Fiction   2809
11                                  Religion   2403
12                                     Women   2291
13                         Fiction / General   2147
14                                 Biography   2061
15                             Short Stories   1911
16                                   History   1869
17                           Crime & Mystery   1860
18          

In [13]:
import numpy as np
clean_df['subjects'] = clean_df['subjects'].replace([None, '<NA>', ''], np.nan)

In [14]:
!pip install torch torchvision torchaudio --quiet
!pip install transformers --quiet

In [15]:
import torch
print("CUDA available:", torch.cuda.is_available())
print("Device name:", torch.cuda.get_device_name(0))


CUDA available: True
Device name: Tesla T4


In [16]:
import json
import ast

path = "/content/genre_keywords.json"
with open(path, "r", encoding="utf-8", errors="ignore") as f:
    raw_text = f.read()


GENRE_KEYWORDS = ast.literal_eval(raw_text)
print(type(GENRE_KEYWORDS))
print(GENRE_KEYWORDS)

<class 'dict'>
{'Fantasy': ['magic', 'dragon', 'wizard', 'elf', 'fairy', 'sorcery', 'spell', 'kingdom', 'orc', 'mythic', 'faerie', 'portal', 'elemental', 'urban fantasy', 'epic fantasy', 'high fantasy', 'dark fantasy'], 'Science Fiction': ['space', 'robot', 'alien', 'dystopian', 'cyberpunk', 'futuristic', 'time travel', 'ai', 'post-apocalyptic', 'interstellar', 'galactic', 'terraform', 'hard sci-fi', 'soft sci-fi', 'space opera'], 'Mystery & Thriller': ['detective', 'murder', 'crime', 'thriller', 'suspense', 'investigation', 'whodunit', 'noir', 'espionage', 'conspiracy', 'cozy mystery', 'psychological thriller', 'legal thriller', 'police', 'private investigators', 'women detectives'], 'Romance': ['romance', 'love', 'affair', 'passion', 'relationship', 'heart', 'bridgerton', 'historical romance', 'contemporary romance', 'romantic suspense', 'erotic', 'paranormal romance', 'marriage'], 'Horror': ['ghost', 'haunted', 'vampire', 'werewolf', 'zombie', 'supernatural', 'demon', 'possession', 

In [18]:
clean_df['subjects'] = clean_df['subjects'].replace('<NA>', pd.NA)

# Filter rows where 'subjects' is null
null_subjects_df = clean_df[clean_df['subjects'].isna()]
null_subjects_df.sample(5)

,ISBN,title,subjects
254621,0394885066,"Good Night, Little Grover (Sesame Street Muppe...",NaN
238621,0440183219,Sprig of Sea Lavender,NaN
205615,0380813297,Ascending,NaN
240198,0448089327,The Crisscross Shadow (Hardy Boys (Hardcover)),NaN
230978,006080484X,Schroeder's Game,NaN


In [19]:
# 1) Load data
titles = null_subjects_df["title"].fillna("").astype(str).tolist()
candidate_labels = list(GENRE_KEYWORDS.keys())

In [20]:
print(titles[:5])
print(candidate_labels[:5])

['C How to Program, 2nd Edition', 'The Mystery Science Theater 3000 Amazing Colossal Episode Guide', 'Nana', 'Le PÃ?Â¨re Goriot', 'La Religieuse']
['Fantasy', 'Science Fiction', 'Mystery & Thriller', 'Romance', 'Horror']


In [21]:
from transformers import pipeline

classifier = pipeline(
    "zero-shot-classification",
    model="typeform/distilbert-base-uncased-mnli",  # lightweight NLI
    framework="pt",    # force PyTorch
    device=0,          # GPU device 0
    batch_size=64
)


/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/776 [00:00<?, ?B/s]

The `xla_device` argument has been deprecated in v4.4.0 of Transformers. It is ignored and you can safely remove it from your `config.json` file.
The `xla_device` argument has been deprecated in v4.4.0 of Transformers. It is ignored and you can safely remove it from your `config.json` file.


model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/258 [00:00<?, ?B/s]

The `xla_device` argument has been deprecated in v4.4.0 of Transformers. It is ignored and you can safely remove it from your `config.json` file.


vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

The `xla_device` argument has been deprecated in v4.4.0 of Transformers. It is ignored and you can safely remove it from your `config.json` file.
The `xla_device` argument has been deprecated in v4.4.0 of Transformers. It is ignored and you can safely remove it from your `config.json` file.
Device set to use cuda:0


In [23]:
# Classify in batches
batch_size = 256
preds = []
for i in tqdm(range(0, len(titles), batch_size)):
    batch = titles[i:i+batch_size]
    outputs = classifier(batch, candidate_labels, multi_label=False)

    # When using batches, outputs is a list of dicts — pick top label per example
    if isinstance(outputs, list):
        preds.extend([out["labels"][0] for out in outputs])
    else:
        preds.append(outputs["labels"][0])


100%|██████████| 267/267 [27:03<00:00,  6.08s/it]


In [42]:
# Concat with main df
clean_df['subjects'] = clean_df['subjects'].replace('<NA>', pd.NA)

null_subjects_df["subjects"] = preds

clean_df = clean_df.drop(index=null_subjects_df.index)

# Append updated rows
clean_df = pd.concat([clean_df, null_subjects_df], ignore_index=True)


In [45]:
clean_df = clean_df.drop_duplicates().reset_index(drop=True)

In [56]:
# Save to CSV
from google.colab import files
clean_df.to_csv("Cleaned_df.csv", index=False)
files.download("Cleaned_df.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [52]:
clean_df.sample(10)

,ISBN,title,subjects
179119,0517100118,Janelle Taylor: Three Complete Novels : Promis...,Historical Fiction | American | Love Stories
170294,1585422606,"Sanity and Grace: A Journey of Suicide, Surviv...",Personal Memoirs | United States | Death
64197,0684839032,VISIONS OF TECHNOLOGY : A Century Of Vital Deb...,Technology
32765,088001654X,Fail Safe,International Relations | Nuclear Weapons | Co...
97761,1888952482,Letters Home: Advice from the Wisest Men and W...,Conduct Of Life | Maxims
165212,0060176652,The Fire Inside: Firefighters Talk About Their...,Fire Fighters
92745,0525481486,The Santa Claus Picture Book: Appraisal Guide,General | Antiques / Collectibles | Children
217124,0689117353,Memoirs of an Invisible Man,Biography & Memoir
139480,076153475X,The Ultimate Code Book: 2001 Edition,Video & Electronic | Games/Puzzles | Games
88057,0849982286,Little David's adventure (Kingdom chums greate...,David | King Of Israel | Goliath


In [51]:
clean_df.isna().sum()

,0
ISBN,0
title,0
subjects,0


In [53]:
clean_df.shape

(270785, 3)

In [54]:
clean_df.duplicated().sum()

np.int64(0)